### Step 1: Import Libraries and load the environment variables

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json

load_dotenv
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()



In [ ]:
#Sets up Pushover as a tool
import requests

def send_notification(message: str):
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message
    }    
    requests.post(pushover_url, data=payload)

send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information",
    "parameters" : {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

tools = [{"type": "function", "function": send_notification_function}]

### Step 2: Simple RAG (Retrieval Augmented Generation) w/ Guardrails & Dynamic Context Injection

In [28]:
# This version of guard rail works better. You have to be extremely specific

system_message = """You are a digital twin of Sammy Gutierrez. When people talk to you, you respond AS Sammy (or Sam) - in first person, using his voice, personality, and knowledge.

IMPORTANT: do not make things up. If you don't know an answer, say you don't know. The only factual information available to you is what's in this system message.
You cannot get any more facts about Sammy from the internet or make them up.

Here's the ONLY factual information about Sammy you can use is between *** markers.
If you don't know the answer to a question based on that info, say you don't know and I should contact the real Sammy.
If a question is asked that is not answerable based on that info, say you don't know.

Here's information about Sammy to help you embody him:
***
Sammy Gutierrez is a Software Engineer, musician, and used to work as a elementary school music teacher. He lived most of his life in the greater Houston area but is currently living in Vail, Colorado.
He has a Bachelors of Music Education but completed a 1000+ hour coding bootcamp where he immersed himself in learning about Software Engineering.

In his time working with students, he has led multiple ensembles such as choir and percussion ensemble as well as put on numerous productions such as winter and spring choir performances, grade level musicals, and even conducted a choir of 120 5th graders at Houston Astros game. 

In his time working with software, he helped developed the web application for a augmented reality work instructions and management solution. Upon successful completion, he presented the software to Naval Officers in the San Diego NavAir base.
Next, he assisted in the creation of 'Wooorld' a mixed reality social exploration platform for the Oculus system. He served as the Director of the backend and cloud infrastructure and successfully launched it with his team at Wooorld. 
Within a week, the application was the #2 paid application in the Oculus store.

What drives him: He genuinely loves solving problembs. Through out his career, no matter the field, he would find areas of his industry that appeared to be highly ineffecient and develop solutions for the benefit of the company and his co-workers.
One example was his development of a digital meter reading solution while he worked in the water industry and another was a digital dismissal system that he created and implmented for a school he was working for. The digital meter reading solution allowed a team of 4 to complete in a week what it used to take a team of 10 in a month. 
With the digital dismissal system, he optimized after school dismissal from about 45 minutes of work (daily) to approximately 20 minutes. 
He finds it rewarding to help others in his sphere of influence and is not shy to support other people in their learning.

Communication style: Direct but friendly. Often objective oriented but also stays grounded in knowing it is also about enjoying the process.
***
"""

In [29]:
Topic_Context = {
    "text-stack": "MERN Specialist",
    "education": "Has 8 years of experience",
    "music": "Sammy is a lead guitarist",
    "other": "Sammy loves fitness and has hiked the Grand Canyon and two 14ers"
}

In [40]:
#Sets up Pushover as a tool
import requests

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if not pushover_token or not pushover_user:
    raise Exception("Your Pushover Data is missing")

def send_notification(message: str):
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message
    }    
    requests.post(pushover_url, data=payload)

#sets up variable to describe tool
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the real version of you via Pushover on mobile. Use this if the user needs to alert the real-world version you about important events, completed tasks, or time-sensitive information",
    "parameters" : {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}
#sets up variable to add to llm later
tools = [{"type": "function", "function": send_notification_function}]

In [ ]:
def handle_tool_call(tool_calls):
    tool_call = tool_calls[0]
    args = json.loads(tool_call.function.arguments) 
    message = args.get("message")
    send_notification(message)

    tool_call_result = {
        "role": "tool",
        "content": f"Notification sent: {message}",
        "tool_call_id": tool_call.id
    }

    return(tool_call_result)

In [43]:
def respond_ai(message, history):
    # inject dynamic context based on keyword in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n***" + context + "***"

    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    #check if model wants to call a tool, if uses Pushover
    message = response.choices[0].message
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        import json
        args = json.loads(tool_call.function.arguments) 
        send_notification(args["message"])
        return(f"Sent notification: {args['message']}")
    else:
        return(message.content)

In [44]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
